# Superstore Dataset Exploratory Data Analysis (EDA)

**Purpose:** Explore and understand distributions, business patterns, anomalies, and relationships in the Superstore Sales dataset prior to pipeline modeling.

*Note: This notebook is purely exploratory. The production pipeline lives in `src/`, `sql/`, and `dags/`.*

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load raw dataset
data_path = Path('../data/raw/superstore.csv')
if not data_path.exists():
    data_path = Path('data/raw/superstore.csv')

df = pd.read_csv(data_path, encoding='windows-1252')
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 1. Schema, Types, and Missing Values Audit

In [ ]:
print("--- Column Data Types ---")
print(df.dtypes)
print("\n--- Missing Values ---")
missing = df.isnull().sum()
print(missing[missing > 0] if (missing > 0).any() else "No missing values found across any columns!")

## 2. Numerical Distributions & Summary Statistics

In [ ]:
num_cols = ['Sales', 'Quantity', 'Discount', 'Profit']
df[num_cols].describe().T

## 3. Anomaly & Loss Detection (Profit Analysis)

Let us observe transactions and products where profit is negative (loss-making operations).

In [ ]:
loss_making_orders = df[df['Profit'] < 0]
print(f"Unprofitable transactions: {len(loss_making_orders)} ({len(loss_making_orders)/len(df):.1%})")
print(f"Total Loss accumulated: ${loss_making_orders['Profit'].sum():,.2f}")

# Top loss-making sub-categories
loss_by_subcat = loss_making_orders.groupby('Sub-Category')['Profit'].sum().sort_values()
print("\n--- Top 5 Loss-Making Sub-Categories ---")
print(loss_by_subcat.head(5))

## 4. Business Patterns: Category & Segment Performance

In [ ]:
cat_summary = df.groupby('Category').agg(
    total_sales=('Sales', 'sum'),
    total_profit=('Profit', 'sum'),
    total_orders=('Order ID', 'nunique')
)
cat_summary['profit_margin'] = (cat_summary['total_profit'] / cat_summary['total_sales']) * 100
cat_summary.sort_values(by='total_sales', ascending=False)

## 5. Temporal Patterns: Yearly and Monthly Trend

In [ ]:
df['Order Date DT'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
df['YearMonth'] = df['Order Date DT'].dt.to_period('M')
monthly_sales = df.groupby('YearMonth').agg(sales=('Sales', 'sum'), profit=('Profit', 'sum'))
print(f"Date Range: {df['Order Date DT'].min().date()} to {df['Order Date DT'].max().date()}")
monthly_sales.tail(12)

## 6. Key Takeaways for Data Engineering Pipeline

1. **Zero Nulls in Raw**: The 9,994 raw records contain zero nulls, but `Postal Code` requires LPAD string conversion to maintain leading zeros.
2. **Product Name Variations**: 32 distinct `Product ID`s contain slight label variations across orders; the Silver transformation deduplicates products deterministically.
3. **Profit Margins**: Overall sales are strongly profitable (12.47% margin), but Tables, Bookcases, and Supplies exhibit recurring losses under heavy discounts (>20%).
4. **Dimensional Modeling**: Clean separation into `silver.customers`, `silver.products`, `silver.geography`, `silver.dim_date`, and `silver.fact_sales` supports high-performance BI reporting.